# Audio analysis notes: segmentation, foreground, and cleanup

These notes use the Billie Eilish tracks as pipeline trials. They are not evidence that the same thresholds or musical assumptions will work for Douyin audio. For Douyin, expect short clips, speech, music, effects, edits, and multiple simultaneous sources.


## How should we define a piece or phase?

There are several different meanings of “segment,” and they should not be conflated:

- **Analysis frame/window:** a small, often overlapping interval (for example, 2 seconds with a 1-second hop) used to calculate features. This gives consistent input units for vectorizing and clustering; it does not claim a musical boundary.
- **Acoustic event/scene boundary:** a point where the sound changes, such as speech starting, music changing, a cut, or a loud effect. Voice activity detection (VAD), change-point detection, or source/event classifiers can help locate these.
- **Musical section/phase:** a verse, chorus, intro, or outro. Repetition and musical structure models can suggest these, but chroma alone cannot reliably name them. Human review or a trained section-labeling model is needed for confident labels.

For an initial Douyin workflow, do not require verse/chorus labels before analysis. First identify speech/music/silence and major acoustic changes; then create fixed or speech-aligned windows for comparable feature vectors. Keep timestamps so any result can be traced back to the original audio. If musical phases matter for a later question, run a separate music-structure step on music-dominant spans.

A practical starting point is 2–5 second windows with overlap, adjusted to the task. Short windows localize changes, while overlap reduces the chance that a key phrase or transition is split exactly at a boundary. For ASR and rhetoric analysis, sentence/utterance spans are often better than arbitrary fixed windows. There is no universally correct duration; check several examples and tune it to the content and model.

## Do the existing segment boundaries work?

The current [segment_boundaries.csv](../results/librosa/segment/segment_boundaries.csv) was produced in the clean-analysis notebook from **chroma self-similarity** using `librosa.segment.agglomerative(chroma, k=8)`. In other words, it proposes coarse divisions based on changes in pitch-class patterns, with the number of segments fixed to eight.

The saved boundary times are approximately 0.00, 3.70, 81.28, 102.82, 141.76, 163.87, 184.80, and 189.40 seconds. They are useful as an exploratory view of musical structure, but they are not yet validated phase labels. The widely varying intervals (including a roughly 77-second span) show that these are not suitable as short, comparable clustering windows. The last recorded point is not the audio end: the track summary reports about 205.93 seconds, so a downstream consumer must explicitly include the final tail through the audio duration.

The method does not detect speech, determine foreground/background, or know whether a boundary is a verse/chorus transition. Its fixed `k=8` is a modeling choice, not a discovered fact. Treat the output as candidate structural boundaries, inspect the recurrence plot and listen around candidate points, and compare against other tracks before using it. For Douyin, chroma can be weak or misleading when clips are speech-led, noisy, heavily edited, or contain little tonal music.

So: **yes, it works as a first coarse music-structure experiment; no, it is not enough as the segmentation stage for propaganda/audio-content clustering.** Build a separate timeline for speech activity and acoustic events, then choose windows suited to the speech or other target content.

## How can a machine tell foreground/main content from background or messy audio?

A machine does not inherently know what is important. “Main content” is a task definition, and the system estimates it from measurable cues and models. Also distinguish **foreground** (prominent or close in the mix) from **important content** (relevant to the research question): quiet speech can be important, while loud music may be background.

Useful signals and tools include:

1. **VAD (voice activity detection):** estimates which time intervals contain speech. It helps find speech candidates, but may miss singing, distant/overlapped speech, or distorted speech, and may mark speech-like sounds as speech.
2. **Audio event/source classification and separation:** estimates speech, music, noise, or sources such as vocals and accompaniment. Separation can make a source easier to analyze, but artifacts are possible; separated stems are estimates, not ground truth. A vocals stem can include singing and can leak music/noise.
3. **Acoustic quality measurements:** clipping, silence, signal-to-noise ratio (SNR), loudness, spectral shape, and abrupt changes can flag technical problems or unusual spans. These are quality cues, not reliable judgments of importance. Loudness alone cannot distinguish speech from loud background music.
4. **ASR and transcript checks:** speech recognition can reveal whether a speech candidate contains words and help align utterances. ASR can hallucinate or fail on music, noise, accents, and overlapping speakers, so retain confidence/timestamps and review uncertain samples.
5. **Human review and task labels:** sample clips should be listened to and labeled with a small, explicit scheme (for example speech, music, mixed, silence/noise, uncertain; plus whether speech is relevant). This is how we measure whether automatic “main content” decisions match the research goal.

A robust first pass is to keep the original audio and create timestamped candidate intervals; add VAD and speech/music estimates; optionally separate sources and run ASR; then assign confidence and review uncertain or high-impact spans. Do not automatically discard a span only because it is quiet, noisy, or classified as background: it may contain relevant speech or a meaningful sound cue. Preserve both the original and processed versions, and log processing choices.

## How to clean up messy audio

First decide what “messy” means for the intended analysis. Technical defects (clipping, hum, wind, low volume) differ from a meaningful mixture (speech over music, several speakers, or sound effects). Cleaning should make the target easier to analyze without erasing evidence.

Suggested order:

1. **Inspect and preserve:** keep an unchanged source copy; record duration, sample rate, channels, and provenance. Listen to representative clips and inspect waveform/spectrogram. Do not overwrite the source.
2. **Standardize safely:** convert to the expected sample rate/channel format for each model, check for empty/corrupt files, and avoid clipping during resampling or level adjustment.
3. **Measure and flag:** detect clipped samples, long silence, very low level, abrupt cuts, and likely low-SNR spans. Use flags and confidence rather than silently deleting them.
4. **Separate the task-relevant source:** use VAD for speech timing, speech enhancement for noisy speech, or source separation when speech and music overlap. Compare enhanced/separated results with the original; processing can distort words and transient sounds.
5. **Transcribe and review:** run ASR on speech candidates, align words/utterances to timestamps, and manually inspect uncertain or consequential spans. Keep “unintelligible/uncertain” as a valid result rather than forcing a guess.
6. **Create analysis windows and vectors:** extract features for each timestamped interval from the original and, where useful, from separated stems. Store provenance and quality indicators alongside every vector.

Avoid treating denoising as neutral: a model may suppress quiet voices, alarms, music, or other meaningful signals. For propaganda research, the content and delivery may both matter, so keep separate representations for speech/text, voice/prosody, music, and overall mix rather than cleaning everything into one signal. Compare downstream results with and without enhancement on a reviewed sample.

## One-sentence summary

Use coarse structure boundaries as hypotheses, not labels; detect and review speech/music/events first, then make timestamped task-appropriate windows and vectorize them while preserving the original audio and uncertainty about what counts as meaningful foreground.

## Work journal / log

| Timestamp (UTC) | Work completed / current understanding |
|---|---|
| 2026-10-03 00:13–00:22 | Inspected the existing `results/` outputs for the Billie Eilish trial: feature summaries, chroma profiles, Demucs vocal/accompaniment stems, Basic Pitch note events, beat tracking, key candidates, and coarse librosa structural segmentation. Discussed moving toward Douyin audio, short-window features/vectorization/clustering, and the need to distinguish speech/content from background and technical noise. |
| 2026-10-03 00:22 | Reviewed the segmentation code and CSV. Confirmed it uses chroma self-similarity with agglomerative segmentation fixed at `k=8`; boundaries are exploratory musical-structure candidates, not short windows, semantic phases, or speech/background labels. Noted variable span lengths and that the last saved boundary (about 189.40 s) precedes the approximately 205.93 s track end. Added these caveats and a practical foreground/cleanup workflow here. |

Add entries with an ISO 8601 UTC timestamp when meaningful project work is completed. Record what was actually done, the data/model involved, and important caveats; do not treat this journal as a substitute for experiment metadata or reproducible logs.